# Deploy pipeline — one-shot end-to-end

Run every cell top-to-bottom to:
1. Verify upstream artifacts (rf preprocess + tabsyn synth)
2. Build `deploy/artifacts/` via `prepare_artifacts.py` (real + TabSyn synth + SMOTE synth + per-year preds)
3. Validate artifact contents
4. (Optional) launch streamlit locally

**Prereq notebooks** (run before this):
- `D:/work/GIS/rf/preprocess/baseline_rf.ipynb` — produces `preprocess_dataset.parquet` + `preds_<year>.parquet` + `synth_tabsyn.parquet` (cell `4cd809f7`)
- `D:/work/GIS/rf/tabsyn/apply_baseline.ipynb` — produces `tabsyn/synthetic/<DATASET>/tabsyn.csv` (optional if pre-baked synth_tabsyn.parquet exists)

Both can be skipped if artifacts already on disk.

## 1. Config

In [ ]:
import os, sys, subprocess, json, shutil
from pathlib import Path

REPO          = Path.cwd()                                   # synth-crop-web/
ARTIFACTS_DIR = REPO / 'deploy' / 'artifacts'
PREP_SCRIPT   = REPO / 'prepare_artifacts.py'

# ---- upstream (rf preprocess + tabsyn) ----
RF_PREP_DIR    = Path(r'D:/work/GIS/rf/preprocess')
RF_DEPLOY_DIR  = RF_PREP_DIR / 'deploy' / 'artifacts'   # has rf_model.joblib + preds_<year>.parquet + synth_tabsyn.parquet
SOURCE_PARQUET = RF_PREP_DIR / 'preprocess_dataset.parquet'
META_JSON      = RF_PREP_DIR / 'preprocess_dataset_meta.json'
TABSYN_PARQ    = RF_DEPLOY_DIR / 'synth_tabsyn.parquet'

# ---- prep params ----
ROWS         = 50_000
WITH_SMOTE   = True
SMOTE_TARGET = 5_000

# ---- streamlit ----
RUN_STREAMLIT = False     # toggle True to launch app at end
STREAMLIT_PORT = 8501

print('REPO        :', REPO)
print('ARTIFACTS   :', ARTIFACTS_DIR)
print('PREP_SCRIPT :', PREP_SCRIPT)
print('UPSTREAM RF :', RF_DEPLOY_DIR)

## 2. Verify upstream artifacts

In [ ]:
checks = [
    ('preprocess_dataset.parquet',   SOURCE_PARQUET),
    ('preprocess_dataset_meta.json', META_JSON),
    ('synth_tabsyn.parquet (pre-baked)', TABSYN_PARQ),
    ('preds_2018.parquet',           RF_DEPLOY_DIR / 'preds_2018.parquet'),
    ('preds_2020.parquet',           RF_DEPLOY_DIR / 'preds_2020.parquet'),
    ('preds_2024.parquet',           RF_DEPLOY_DIR / 'preds_2024.parquet'),
    ('rf_model.joblib (cascade)',    RF_DEPLOY_DIR / 'rf_model.joblib'),
]
missing = []
for label, path in checks:
    ok = path.exists()
    sz = f'{path.stat().st_size/1e6:>7.2f} MB' if ok else '       -'
    print(f'  [{"OK" if ok else "MISS"}] {label:<40s} {sz}  {path}')
    if not ok:
        missing.append(label)

if missing:
    raise FileNotFoundError(
        f'missing upstream artifacts: {missing}\n'
        f'-> run baseline_rf.ipynb cell 4cd809f7 + apply_baseline.ipynb cell da43b870 first'
    )
print('\nALL UPSTREAM ARTIFACTS PRESENT')

## 3. Run prepare_artifacts.py

Produces `deploy/artifacts/`:
- 8 required files (dataset / class_counts / feature_stats / corr_real / metrics / confusion / feature_importance / rf_model)
- 5 optional (synth_tabsyn / corr_syn / synth_smote / corr_smote / grid_meta)
- 3 preds_<year>.parquet (true vs RF-cascade-pred per labelled pixel)

In [ ]:
cmd = [sys.executable, str(PREP_SCRIPT),
       '--source-csv', str(SOURCE_PARQUET),
       '--synth-csv',  str(TABSYN_PARQ),
       '--meta-json',  str(META_JSON),
       '--preds-dir',  str(RF_DEPLOY_DIR),
       '--rows',       str(ROWS)]
if WITH_SMOTE:
    cmd += ['--with-smote', '--smote-target', str(SMOTE_TARGET)]

print('$', ' '.join(cmd))
env = dict(os.environ); env.setdefault('PYTHONUNBUFFERED', '1'); env.setdefault('PYTHONIOENCODING', 'utf-8')
p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=subprocess.PIPE,
                     stderr=subprocess.STDOUT, text=True, bufsize=1, encoding='utf-8', errors='replace')
for line in p.stdout: print(line, end='', flush=True)
p.wait()
if p.returncode != 0:
    raise RuntimeError(f'prepare_artifacts.py failed (exit {p.returncode})')
print('\nPREP DONE')

## 4. Verify artifacts produced

In [ ]:
import pandas as pd, numpy as np, joblib

REQUIRED = [
    'dataset.parquet', 'class_counts.parquet', 'feature_stats.parquet',
    'corr_real.npy', 'metrics.json', 'confusion.npy',
    'feature_importance.parquet', 'rf_model.joblib',
]
OPTIONAL = [
    'synth_tabsyn.parquet', 'corr_syn.npy',
    'synth_smote.parquet',  'corr_smote.npy',
    'grid_meta.json', 'class_counts_year.parquet',
]
PREDS = sorted(ARTIFACTS_DIR.glob('preds_*.parquet'))
RASTERS = sorted(ARTIFACTS_DIR.glob('preds_*.tif'))

print('=== required ===')
missing_req = []
for f in REQUIRED:
    p = ARTIFACTS_DIR / f
    ok = p.exists()
    sz = f'{p.stat().st_size/1e6:>7.2f} MB' if ok else '       -'
    print(f'  [{"OK" if ok else "MISS"}] {f:<32s} {sz}')
    if not ok: missing_req.append(f)

print('\n=== optional ===')
for f in OPTIONAL:
    p = ARTIFACTS_DIR / f
    ok = p.exists()
    sz = f'{p.stat().st_size/1e6:>7.2f} MB' if ok else '       -'
    print(f'  [{"OK" if ok else "miss"}] {f:<32s} {sz}')

print(f'\n=== preds_<year>.parquet (Segmentation page) ===')
for p in PREDS:
    print(f'  [OK] {p.name:<32s} {p.stat().st_size/1e6:>7.2f} MB')
if not PREDS:
    print('  [miss] none — Segmentation page will be empty')

if RASTERS:
    print(f'\n=== preds_<year>.tif (raster overlays) ===')
    for r in RASTERS: print(f'  [OK] {r.name:<32s} {r.stat().st_size/1e6:>7.2f} MB')

if missing_req:
    raise RuntimeError(f'REQUIRED files missing: {missing_req}')

total_mb = sum(f.stat().st_size for f in ARTIFACTS_DIR.glob('*') if f.is_file()) / 1e6
print(f'\nTOTAL: {total_mb:.1f} MB')

## 5. Smoke-test artifact loads (no streamlit)

In [ ]:
ds = pd.read_parquet(ARTIFACTS_DIR / 'dataset.parquet')
print(f'dataset.parquet  : {ds.shape}  cols={list(ds.columns)[:6]}...')
print(f'  label range: {ds["label"].min()}..{ds["label"].max()}')
print(f'  classes:    {sorted(ds["label"].unique())}')
if "year" in ds.columns:
    print(f'  years:      {sorted(ds["year"].unique())}')

metrics = json.loads((ARTIFACTS_DIR / 'metrics.json').read_text(encoding='utf-8'))
print(f'\nmetrics.json     : keys={list(metrics.keys())}')
print(f'  acc       : {metrics["metrics"]["accuracy"]:.4f}')
print(f'  kappa     : {metrics["metrics"]["kappa"]:.4f}')
print(f'  f1_macro  : {metrics["metrics"]["f1_macro"]:.4f}')
print(f'  f1_weighted: {metrics["metrics"]["f1_weighted"]:.4f}')

cm = np.load(ARTIFACTS_DIR / 'confusion.npy')
print(f'\nconfusion.npy    : shape={cm.shape}  diag_sum={int(np.diag(cm).sum()):,}/{int(cm.sum()):,}')

rf = joblib.load(ARTIFACTS_DIR / 'rf_model.joblib')
print(f'\nrf_model.joblib  : type={type(rf).__name__}  classes={getattr(rf, "classes_", None)}')

if PREDS:
    p = pd.read_parquet(PREDS[0])
    agree = float((p["label_true"] == p["label_pred"]).mean())
    print(f'\n{PREDS[0].name:<24s}: {len(p):,} rows  pixel_agree={agree:.4f}')

print('\nALL ARTIFACTS LOAD OK')

## 6. (Optional) Launch streamlit locally

Set `RUN_STREAMLIT = True` in cell §1 to launch. Else copy command into terminal.

In [ ]:
stream_cmd = ['streamlit', 'run', str(REPO / 'deploy' / 'streamlit_app.py'),
              '--server.port', str(STREAMLIT_PORT), '--server.headless', 'true']
print('streamlit launch command:')
print('  $', ' '.join(stream_cmd))
print(f'  url:      http://localhost:{STREAMLIT_PORT}')

if RUN_STREAMLIT:
    print(f'\nlaunching... (Ctrl+C to stop)')
    subprocess.run(stream_cmd, cwd=str(REPO))

## 7. Cloud deploy (Streamlit Community Cloud)

**Option A — un-ignore artifacts** (simplest):
1. Edit `.gitignore`: comment out `deploy/artifacts/*`.
2. `git add deploy/artifacts && git commit -m 'snapshot demo artifacts' && git push`.
3. Streamlit Cloud → New app → point at `deploy/streamlit_app.py`.

**Option B — release-asset upload** (if artifacts > 100 MB):
1. `cd deploy/artifacts && zip ../artifacts.zip . && cd ../..`.
2. Upload `artifacts.zip` as GitHub release asset.
3. Add boot-time downloader to `streamlit_app.py` (fetch + extract on first run).

**Pinned requirements** (avoid breaking-version drift on cloud):
```
pip freeze > deploy/requirements.txt
```

In [ ]:
# show artifact size + suggest deploy strategy
total_mb = sum(f.stat().st_size for f in ARTIFACTS_DIR.glob('*') if f.is_file()) / 1e6
GH_LIMIT_MB = 100
STC_LIMIT_GB = 1
print(f'artifacts total: {total_mb:.1f} MB')
if total_mb < GH_LIMIT_MB:
    print(f'  -> under {GH_LIMIT_MB} MB: safe to commit (Option A)')
elif total_mb < STC_LIMIT_GB * 1024:
    print(f'  -> {GH_LIMIT_MB}-{STC_LIMIT_GB*1024} MB: too big for git, use release asset (Option B)')
else:
    print(f'  -> over Streamlit Cloud {STC_LIMIT_GB} GB limit. Slim:')
    print(f'     - downsample preds_<year>.parquet (every Nth pixel)')
    print(f'     - lower --rows in §1 (current: {ROWS:,})')
    print(f'     - drop --with-smote')